# LightOnOCR-2 Magyar Fine-tuning (v12)

**Runtime → Change runtime type → T4 GPU**

Memória-optimalizált verzió

In [ ]:
# 1. Telepítés + Fontok
!pip install -q transformers>=4.45.0 peft datasets accelerate pillow opencv-python-headless

!mkdir -p /content/fonts

print('Fontok letöltése...')
!wget -q 'https://github.com/liberationfonts/liberation-fonts/files/7261482/liberation-fonts-ttf-2.1.5.tar.gz' -O /tmp/lib.tar.gz && tar -xzf /tmp/lib.tar.gz -C /tmp/ && cp /tmp/liberation-fonts-ttf-2.1.5/*.ttf /content/fonts/
!wget -q 'https://github.com/dejavu-fonts/dejavu-fonts/releases/download/version_2_37/dejavu-fonts-ttf-2.37.zip' -O /tmp/dv.zip && unzip -q -o /tmp/dv.zip -d /tmp/ && cp /tmp/dejavu-fonts-ttf-2.37/ttf/*.ttf /content/fonts/

!ls /content/fonts/*.ttf | wc -l

In [ ]:
# 2. Font teszt
import os, glob, random, json
from PIL import Image, ImageDraw, ImageFont, ImageFilter
import numpy as np
from pathlib import Path
from IPython.display import display

def test_font(fp):
    try:
        font = ImageFont.truetype(fp, 28)
        img = Image.new('RGB', (300, 40), 'white')
        ImageDraw.Draw(img).text((5, 5), 'őűŐŰ éáí', fill='black', font=font)
        return np.sum(np.array(img) < 100) > 80, img
    except: return False, None

FONTS = []
for fp in sorted(glob.glob('/content/fonts/*.ttf')):
    ok, img = test_font(fp)
    if ok:
        FONTS.append((os.path.basename(fp).replace('.ttf',''), fp))
        if len(FONTS) <= 3: display(img)

print(f'{len(FONTS)} font OK')

In [ ]:
# 3. Adatgenerálás - KISEBB KÉPEK
IMG_W, IMG_H = 600, 300  # Kisebb képek = kevesebb memória

WORDS = ['őr','őriz','ők','ősz','erő','idő','mező','tető','fő','nő',
         'belső','külső','felső','alsó','első','költő','vezető',
         'űr','gyűrű','tűz','fűz','hűtő','hűség','szűk','sűrű',
         'fizetendő','összeg','adószám','díj','működik']

def gen_text():
    lines = [' '.join(random.sample(WORDS, 5))]
    lines.append(f'Összeg: {random.randint(1,99)} {random.randint(100,999):03d} Ft')
    lines.append(f'Adószám: {random.randint(10000000,99999999)}-{random.randint(1,2)}-{random.randint(10,99)}')
    lines.append('őűŐŰ öüóéáíú')
    lines.append(' '.join(random.sample(WORDS, 4)))
    return '\n'.join(lines)

def render(text, font_path, size=22):
    img = Image.new('RGB', (IMG_W, IMG_H), 'white')
    draw = ImageDraw.Draw(img)
    font = ImageFont.truetype(font_path, size)
    y = 25
    for line in text.split('\n'):
        draw.text((25, y), line, fill='black', font=font)
        y += int(size * 1.4)
    return img

Path('data/images').mkdir(parents=True, exist_ok=True)
N = 400  # Kevesebb kép

print(f'Generálás: {N} kép ({IMG_W}x{IMG_H})...')
annotations = []
for i in range(N):
    text = gen_text()
    _, fpath = random.choice(FONTS)
    img = render(text, fpath, random.choice([20,22,24]))
    if random.random() < 0.3:
        img = img.rotate(random.uniform(-1.5, 1.5), fillcolor='white')
    img.save(f'data/images/{i:04d}.png')
    annotations.append({'image': f'{i:04d}.png', 'text': text})
    if (i+1) % 100 == 0: print(f'  {i+1}/{N}')

with open('data/ann.jsonl', 'w', encoding='utf-8') as f:
    for a in annotations:
        f.write(json.dumps(a, ensure_ascii=False) + '\n')

print(f'✓ {N} kép')
display(Image.open('data/images/0000.png'))

In [ ]:
# 4. Modell - MEMÓRIA OPTIMALIZÁLÁS
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText
from peft import LoraConfig, get_peft_model

# GPU memória ürítése
torch.cuda.empty_cache()

MODEL_ID = 'lightonai/LightOnOCR-2-1B-base'
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, 
    torch_dtype=torch.bfloat16,
    device_map='auto',
    low_cpu_mem_usage=True,
)
processor = AutoProcessor.from_pretrained(MODEL_ID)

# Gradient checkpointing - memória csökkentés
model.gradient_checkpointing_enable()

# Kisebb LoRA rank
model = get_peft_model(model, LoraConfig(
    r=8,  # Kisebb rank
    lora_alpha=16,
    target_modules=['q_proj','v_proj'],  # Kevesebb modul
    lora_dropout=0.05
))
model.print_trainable_parameters()

print(f'GPU memória: {torch.cuda.memory_allocated()/1e9:.1f} GB')

In [ ]:
# 5. Dataset
from torch.utils.data import Dataset as TorchDataset

class OCRDataset(TorchDataset):
    def __init__(self, jsonl_path, img_dir, processor):
        self.processor = processor
        self.img_dir = img_dir
        self.data = [json.loads(l) for l in open(jsonl_path, encoding='utf-8')]
    
    def __len__(self): return len(self.data)
    
    def __getitem__(self, idx):
        item = self.data[idx]
        img = Image.open(f"{self.img_dir}/{item['image']}").convert('RGB')
        img_in = self.processor.image_processor(img, return_tensors='pt')
        txt_in = self.processor.tokenizer(item['text'], return_tensors='pt', padding='max_length', max_length=256, truncation=True)
        return {
            'pixel_values': img_in['pixel_values'].squeeze(0),
            'input_ids': txt_in['input_ids'].squeeze(0),
            'attention_mask': txt_in['attention_mask'].squeeze(0),
            'labels': txt_in['input_ids'].squeeze(0),
        }

dataset = OCRDataset('data/ann.jsonl', 'data/images', processor)
print(f'✓ {len(dataset)} kép, pixel: {dataset[0]["pixel_values"].shape}')

In [ ]:
# 6. Training - MEMÓRIA OPTIMALIZÁLT
from transformers import TrainingArguments, Trainer

torch.cuda.empty_cache()

args = TrainingArguments(
    output_dir='./lora',
    num_train_epochs=3,
    per_device_train_batch_size=1,  # Batch size 1!
    gradient_accumulation_steps=16,  # Kompenzálás
    learning_rate=5e-5,
    warmup_ratio=0.1,
    logging_steps=10,
    save_steps=50,
    bf16=True,
    remove_unused_columns=False,
    report_to='none',
    gradient_checkpointing=True,  # Memória csökkentés
    optim='adamw_torch_fused',  # Gyorsabb optimizer
)

trainer = Trainer(model=model, args=args, train_dataset=dataset)
print(f'Tanítás ({len(dataset)} kép, batch=1, grad_accum=16)...')
trainer.train()
print('✓ Kész!')

In [ ]:
# 7. Mentés
model.save_pretrained('./lora')
merged = model.merge_and_unload()
merged.save_pretrained('./merged')
processor.save_pretrained('./merged')
print('✓ Mentve')

In [ ]:
# 8. Teszt
torch.cuda.empty_cache()
for idx in [0, 100, 200]:
    img = Image.open(f'data/images/{idx:04d}.png')
    inputs = processor.image_processor(img, return_tensors='pt')
    inputs = {k: v.to(merged.device) for k, v in inputs.items()}
    inputs['input_ids'] = processor.tokenizer('', return_tensors='pt')['input_ids'].to(merged.device)
    with torch.no_grad():
        out = merged.generate(**inputs, max_new_tokens=300, do_sample=False)
    print(f'\n=== #{idx} ===')
    display(img)
    print(processor.tokenizer.decode(out[0], skip_special_tokens=True))

In [ ]:
# 9. Letöltés
!zip -r merged.zip merged/
from google.colab import files
files.download('merged.zip')
print('\nMAC: unzip merged.zip && mlx_vlm convert --hf-path merged --mlx-path lighton-hun-mlx -q --q-bits 4')